# 05 - Table 2: target awareness by docking

Paper protocol (Section 3.2.2): for five novel proteins with resolved binding sites,
report the percentage of generated molecules docking below **-7.0 kcal/mol**, with
**SMINA at default settings** and a docking box **5 A larger than the co-crystal
ligand**. Averaged over 10 runs of 100 molecules.

**External requirements** - this script does nothing useful without them:
- `smina` on `PATH`
- the five receptor PDBs, each with its co-crystal ligand still in the file
  (`--autobox_ligand` needs it)

**Before trusting any number here, check the PDB -> protein-row mapping.** The paper
docks molecules generated *for that protein*; if the mapping is wrong you are docking
molecules conditioned on something else. The matcher below is a helper, not an oracle:
BindingDB carries full UniProt sequences while a PDB entry is a construct, so the match
is partial by construction. Read the report it prints.

**Cost**: 5 proteins x 10 runs x ~95 valid molecules is roughly 4,750 dockings. At
default exhaustiveness that is many CPU-hours. Start with `DOCK_RUNS = [1]` and a small
`MAX_LIGANDS_PER_CELL` to time it, then scale.

In [1]:
import os
import re
import csv
import glob
import json
import time
import shutil
import difflib
import subprocess
from collections import defaultdict

import numpy as np
import pandas as pd

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog('rdApp.*')

## Config

In [2]:
REPO_ROOT = os.environ.get("CTDDG_REPO_ROOT", "/home/muhamed/repo/CTDDG")
FOLD = os.environ.get("CTDDG_FOLD", "d1")

SAMPLES_DIR = os.path.join(REPO_ROOT, "outputs/samples", FOLD)
BINDINGDB_DIR = os.path.join(REPO_ROOT, "datasets/data/bindingdb")
EVAL_DIR = os.path.join(REPO_ROOT, "outputs/evaluation", FOLD)
DOCK_DIR = os.path.join(EVAL_DIR, "docking")
RECEPTOR_DIR = os.path.join(REPO_ROOT, "datasets/receptors")
PDB_MAP_PATH = os.path.join(EVAL_DIR, "pdb_protein_map.json")

os.makedirs(DOCK_DIR, exist_ok=True)
os.makedirs(RECEPTOR_DIR, exist_ok=True)

FOLD_SUFFIX = {"d1": "4_org_1042_104", "d2": "4_org_1000_112", "d3": "4_org_1004_122",
               "d4": "4_org_1002_103", "d5": "4_org_1036_124"}

# CDGCN / CTDDG Table 2 targets.
TARGETS = {
    "4nos": "Nitric oxide synthase, inducible",
    "2nru": "Interleukin-1 receptor-associated kinase 4",
    "1lqf": "Tyrosine-protein phosphatase non-receptor type 1",
    "1k2r": "Nitric oxide synthase, brain",
    "1cqp": "Integrin alpha-L",
}

AFFINITY_THRESHOLD = -7.0
AUTOBOX_ADD = 5.0            # paper: box 5 A larger than the ligand
SMINA_CPU = int(os.environ.get("CTDDG_SMINA_CPU", "8"))
SMINA_SEED = 42              # deviation from "defaults", for reproducibility
DOCK_RUNS = None             # None = every run present; e.g. [1] to start
MAX_LIGANDS_PER_CELL = None  # None = all unique valid; e.g. 20 for a timing test
OVERWRITE = False

# Set manually to override the sequence matcher, e.g. {"4nos": 17, "2nru": 63}.
MANUAL_PDB_MAP = {}

PAPER_TABLE2_CTDDG = {'4nos': (95.55, 2.56), '2nru': (93.33, 3.58), '1lqf': (98.03, 2.09),
                      '1k2r': (95.71, 2.92), '1cqp': (86.35, 3.43), 'Avg': (93.80, 2.92)}
PAPER_TABLE2_CDGCN = {'4nos': (99.10, 1.08), '2nru': (96.28, 2.15), '1lqf': (96.02, 1.51),
                      '1k2r': (98.57, 0.67), '1cqp': (85.00, 4.07), 'Avg': (95.00, 1.90)}

SMINA = shutil.which("smina")
print(f"smina: {SMINA or 'NOT FOUND - install it or add it to PATH'}")

smina: NOT FOUND - install it or add it to PATH


## Map each PDB to its protein row in the held-out split

Row index = line number in `<fold>_te_unique_proteins.txt` = HDF5 key `p{row+1}` =
the row of the embedding matrix the molecules were conditioned on.

In [3]:
AA3TO1 = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q', 'GLU': 'E',
    'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F',
    'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
    'MSE': 'M', 'SEC': 'U', 'PYL': 'O',
}


def receptor_path(pdb):
    for ext in ('.pdb', '.ent'):
        p = os.path.join(RECEPTOR_DIR, f"{pdb}{ext}")
        if os.path.isfile(p):
            return p
    return None


def pdb_seqres(path):
    """{chain: one-letter sequence} from SEQRES, falling back to ATOM CA records."""
    chains = defaultdict(list)
    with open(path) as f:
        lines = f.readlines()

    for ln in lines:
        if ln.startswith("SEQRES"):
            chain = ln[11]
            chains[chain] += [AA3TO1.get(r, 'X') for r in ln[19:].split()]
    if chains:
        return {c: ''.join(s) for c, s in chains.items()}

    seen = set()
    for ln in lines:
        if ln.startswith("ATOM") and ln[12:16].strip() == "CA":
            key = (ln[21], ln[22:27])
            if key in seen:
                continue
            seen.add(key)
            chains[ln[21]].append(AA3TO1.get(ln[17:20].strip(), 'X'))
    return {c: ''.join(s) for c, s in chains.items()}


def read_protein_list():
    path = os.path.join(BINDINGDB_DIR, "test_dataset", f"test_{FOLD_SUFFIX[FOLD]}",
                        f"{FOLD}_te_unique_proteins.txt")
    with open(path) as f:
        return [''.join(ln.split()).upper() for ln in f if ln.strip()]


def match_sequence(query, candidates, top_k=3):
    """Rank held-out proteins against a PDB chain sequence.

    `contained` - a 40-residue window of the PDB chain found verbatim in the candidate.
    That is the strong signal; `ratio` alone is misleading because a PDB construct is a
    fragment of the UniProt sequence BindingDB carries.
    """
    q = query.replace('X', '')
    probes = [q[i:i + 40] for i in range(0, max(1, len(q) - 40), 40)][:8]
    scored = []
    for i, cand in enumerate(candidates):
        hits = sum(1 for p in probes if p and p in cand)
        ratio = difflib.SequenceMatcher(None, q[:400], cand[:400]).ratio()
        scored.append((hits / max(1, len(probes)), ratio, i))
    scored.sort(reverse=True)
    return [{'row': i, 'contained': round(h, 2), 'ratio': round(r, 3),
             'cand_len': len(candidates[i])} for h, r, i in scored[:top_k]]


PROTEINS = read_protein_list()
print(f"{len(PROTEINS)} held-out proteins in fold {FOLD}\n")

pdb_map, mapping_report = {}, []
for pdb in TARGETS:
    if pdb in MANUAL_PDB_MAP:
        pdb_map[pdb] = int(MANUAL_PDB_MAP[pdb])
        mapping_report.append({'PDB': pdb, 'row': pdb_map[pdb], 'source': 'manual',
                               'contained': '-', 'ratio': '-'})
        continue

    rp = receptor_path(pdb)
    if rp is None:
        mapping_report.append({'PDB': pdb, 'row': None, 'source': 'no receptor file',
                               'contained': '-', 'ratio': '-'})
        continue

    best_overall = None
    for chain, seq in pdb_seqres(rp).items():
        if len(seq.replace('X', '')) < 50:
            continue
        for cand in match_sequence(seq, PROTEINS, top_k=1):
            if best_overall is None or (cand['contained'], cand['ratio']) > \
                    (best_overall['contained'], best_overall['ratio']):
                best_overall = {**cand, 'chain': chain}

    if best_overall is None:
        mapping_report.append({'PDB': pdb, 'row': None, 'source': 'no usable chain',
                               'contained': '-', 'ratio': '-'})
        continue

    pdb_map[pdb] = best_overall['row']
    mapping_report.append({'PDB': pdb, 'row': best_overall['row'],
                           'source': f"chain {best_overall['chain']}",
                           'contained': best_overall['contained'],
                           'ratio': best_overall['ratio']})

print(pd.DataFrame(mapping_report).to_string(index=False))
print("\nCheck this. `contained` near 1.0 is a confident match; near 0 with a middling "
      "`ratio` is not, and you should set MANUAL_PDB_MAP instead of docking on a guess.")

if pdb_map:
    with open(PDB_MAP_PATH, 'w') as f:
        json.dump(pdb_map, f, indent=1)
    print(f"\nwrote {PDB_MAP_PATH} (04_evaluate_metrics.py reads it for the Table 3 rows)")

104 held-out proteins in fold d1

 PDB  row           source contained ratio
4nos None no receptor file         -     -
2nru None no receptor file         -     -
1lqf None no receptor file         -     -
1k2r None no receptor file         -     -
1cqp None no receptor file         -     -

Check this. `contained` near 1.0 is a confident match; near 0 with a middling `ratio` is not, and you should set MANUAL_PDB_MAP instead of docking on a guess.


## Receptor and reference ligand

`--autobox_ligand` needs the co-crystal ligand as its own file. Pulled out of the PDB
as the largest non-water, non-ion HETATM residue.

In [4]:
SKIP_HET = {'HOH', 'WAT', 'SO4', 'PO4', 'GOL', 'EDO', 'NA', 'CL', 'MG', 'ZN', 'CA',
            'K', 'MN', 'FE', 'ACT', 'PEG', 'DMS', 'TRS', 'MPD', 'IOD', 'BR', 'NO3'}


def prepare_receptor(pdb):
    """-> (receptor_only.pdb, reference_ligand.pdb) or (None, None)."""
    src = receptor_path(pdb)
    if src is None:
        return None, None

    rec_out = os.path.join(DOCK_DIR, f"{pdb}_receptor.pdb")
    lig_out = os.path.join(DOCK_DIR, f"{pdb}_refligand.pdb")
    if os.path.isfile(rec_out) and os.path.isfile(lig_out) and not OVERWRITE:
        return rec_out, lig_out

    with open(src) as f:
        lines = f.readlines()

    het = defaultdict(list)
    for ln in lines:
        if ln.startswith("HETATM"):
            resn = ln[17:20].strip()
            if resn in SKIP_HET:
                continue
            het[(resn, ln[21], ln[22:27].strip())].append(ln)
    if not het:
        print(f"[{pdb}] no co-crystal ligand found - cannot autobox. Supply a box manually.")
        return None, None

    key = max(het, key=lambda k: len(het[k]))
    with open(lig_out, 'w') as f:
        f.writelines(het[key])
        f.write("END\n")
    with open(rec_out, 'w') as f:
        f.writelines([ln for ln in lines if ln.startswith(("ATOM", "TER"))])
        f.write("END\n")
    print(f"[{pdb}] receptor + reference ligand {key[0]} (chain {key[1]}, "
          f"{len(het[key])} atoms)")
    return rec_out, lig_out

## Ligand 3D preparation

In [5]:
def smiles_to_sdf(smiles_list, out_path, seed=42):
    """ETKDGv3 embed + MMFF optimise. -> list of names actually written."""
    writer = Chem.SDWriter(out_path)
    names = []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        mol = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = seed + i
        if AllChem.EmbedMolecule(mol, params) != 0:
            params.useRandomCoords = True
            if AllChem.EmbedMolecule(mol, params) != 0:
                continue
        try:
            AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
        except Exception:
            pass
        name = f"lig{i:04d}"
        mol.SetProp("_Name", name)
        mol.SetProp("SMILES", smi)
        writer.write(mol)
        names.append(name)
    writer.close()
    return names

## Dock

In [6]:
def run_smina(receptor, ref_ligand, ligands_sdf, out_sdf, log_path):
    cmd = [SMINA, "-r", receptor, "-l", ligands_sdf,
           "--autobox_ligand", ref_ligand, "--autobox_add", str(AUTOBOX_ADD),
           "-o", out_sdf, "--log", log_path,
           "--cpu", str(SMINA_CPU), "--seed", str(SMINA_SEED)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "").strip().splitlines()[-8:]
        return False, "\n    ".join(tail)
    return True, "ok"


def parse_affinities(out_sdf):
    """-> {ligand name: best (most negative) affinity}."""
    best = {}
    supplier = Chem.SDMolSupplier(out_sdf, sanitize=False, removeHs=False)
    for mol in supplier:
        if mol is None:
            continue
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else None
        aff = None
        for tag in ("minimizedAffinity", "CNNaffinity", "affinity"):
            if mol.HasProp(tag):
                try:
                    aff = float(mol.GetProp(tag))
                    break
                except ValueError:
                    pass
        if name is None or aff is None:
            continue
        best[name] = min(aff, best.get(name, np.inf))
    return best


def load_cell(run, protein_row):
    path = os.path.join(SAMPLES_DIR, f"run{run}", f"protein_{protein_row:04d}.csv")
    if not os.path.isfile(path):
        return []
    with open(path) as f:
        smis = [r['smiles'] for r in csv.DictReader(f) if r['smiles']]
    valid = [s for s in smis if Chem.MolFromSmiles(s) is not None]
    return list(dict.fromkeys(valid))


def dock_cell(pdb, protein_row, run, receptor, ref_ligand):
    cell_dir = os.path.join(DOCK_DIR, pdb, f"run{run}")
    os.makedirs(cell_dir, exist_ok=True)
    results_csv = os.path.join(cell_dir, "affinities.csv")
    if os.path.isfile(results_csv) and not OVERWRITE:
        return pd.read_csv(results_csv)

    smiles = load_cell(run, protein_row)
    if MAX_LIGANDS_PER_CELL:
        smiles = smiles[:MAX_LIGANDS_PER_CELL]
    if not smiles:
        return None

    lig_sdf = os.path.join(cell_dir, "ligands.sdf")
    out_sdf = os.path.join(cell_dir, "docked.sdf")
    names = smiles_to_sdf(smiles, lig_sdf)
    if not names:
        print(f"  [{pdb} run{run}] no ligand embedded in 3D")
        return None

    t0 = time.time()
    ok, msg = run_smina(receptor, ref_ligand, lig_sdf,
                        out_sdf, os.path.join(cell_dir, "smina.log"))
    if not ok:
        print(f"  [{pdb} run{run}] smina failed:\n    {msg}")
        return None

    aff = parse_affinities(out_sdf)
    rows = [{'name': f"lig{i:04d}", 'smiles': s,
             'affinity': aff.get(f"lig{i:04d}", np.nan)}
            for i, s in enumerate(smiles) if f"lig{i:04d}" in names]
    df = pd.DataFrame(rows)
    df.to_csv(results_csv, index=False)
    print(f"  [{pdb} run{run}] {df['affinity'].notna().sum()}/{len(df)} docked in "
          f"{time.time() - t0:.0f}s | best {df['affinity'].min():.2f}")
    return df

In [7]:
runs_available = sorted(
    int(re.search(r"run(\d+)$", d).group(1))
    for d in glob.glob(os.path.join(SAMPLES_DIR, "run*")) if os.path.isdir(d))
runs = DOCK_RUNS or runs_available
print(f"runs to dock: {runs}")

records = []
if SMINA is None:
    print("\n[BLOCKED] smina not on PATH. Install it, then re-run:\n"
          "  conda install -c conda-forge smina        # or build from source")
elif not pdb_map:
    print("\n[BLOCKED] no PDB mapped. Download the receptors into "
          f"{RECEPTOR_DIR}:\n"
          "  for p in 4nos 2nru 1lqf 1k2r 1cqp; do \\\n"
          f"    wget -O {RECEPTOR_DIR}/$p.pdb https://files.rcsb.org/download/${{p^^}}.pdb; \\\n"
          "  done\n"
          "Keep the co-crystal ligand in the file - --autobox_ligand needs it.")
else:
    for pdb, protein_row in pdb_map.items():
        receptor, ref_ligand = prepare_receptor(pdb)
        if receptor is None:
            continue
        print(f"\n{pdb} ({TARGETS[pdb]}) -> protein row {protein_row}")
        for run in runs:
            df = dock_cell(pdb, protein_row, run, receptor, ref_ligand)
            if df is None or df.empty:
                continue
            scored = df['affinity'].dropna()
            records.append({
                'pdb': pdb, 'run': run, 'protein_row': protein_row,
                'n_docked': len(scored),
                'pct_below_threshold': 100.0 * (scored < AFFINITY_THRESHOLD).mean(),
                'best_affinity': float(scored.min()),
                'mean_affinity': float(scored.mean()),
            })

runs to dock: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

[BLOCKED] smina not on PATH. Install it, then re-run:
  conda install -c conda-forge smina        # or build from source


## Table 2

In [8]:
if records:
    DOCK = pd.DataFrame(records)
    DOCK.to_csv(os.path.join(EVAL_DIR, "docking_per_pdb_per_run.csv"), index=False)

    rows = []
    for pdb in TARGETS:
        sub = DOCK[DOCK['pdb'] == pdb]
        if sub.empty:
            continue
        v = sub.groupby('run')['pct_below_threshold'].mean()
        mu = float(v.mean())
        sd = float(v.std(ddof=1)) if len(v) > 1 else 0.0
        rows.append({
            'PDB': pdb,
            'This run (%)': f"{mu:.2f} +/- {sd:.2f}",
            'CTDDG paper': f"{PAPER_TABLE2_CTDDG[pdb][0]} +/- {PAPER_TABLE2_CTDDG[pdb][1]}",
            'CDGCN paper': f"{PAPER_TABLE2_CDGCN[pdb][0]} +/- {PAPER_TABLE2_CDGCN[pdb][1]}",
            'Best affinity': f"{sub['best_affinity'].min():.2f}",
            '_mu': mu, '_sd': sd,
        })

    if rows:
        avg_mu = float(np.mean([r['_mu'] for r in rows]))
        avg_sd = float(np.mean([r['_sd'] for r in rows]))
        rows.append({
            'PDB': 'Avg',
            'This run (%)': f"{avg_mu:.2f} +/- {avg_sd:.2f}",
            'CTDDG paper': f"{PAPER_TABLE2_CTDDG['Avg'][0]} +/- {PAPER_TABLE2_CTDDG['Avg'][1]}",
            'CDGCN paper': f"{PAPER_TABLE2_CDGCN['Avg'][0]} +/- {PAPER_TABLE2_CDGCN['Avg'][1]}",
            'Best affinity': '-', '_mu': avg_mu, '_sd': avg_sd,
        })

        TABLE2 = pd.DataFrame(rows).drop(columns=['_mu', '_sd'])
        print(f"\n=== Table 2: % of generated molecules below "
              f"{AFFINITY_THRESHOLD} kcal/mol ===")
        print(TABLE2.to_string(index=False))
        TABLE2.to_csv(os.path.join(EVAL_DIR, "table2_target_awareness.csv"), index=False)

        with open(os.path.join(EVAL_DIR, "report.md"), 'a') as f:
            f.write("\n## Table 2 - target awareness (docking)\n\n")
            f.write(TABLE2.to_markdown(index=False))
            f.write(f"\n\nSMINA, autobox +{AUTOBOX_ADD} A on the co-crystal ligand, "
                    f"seed {SMINA_SEED}, {SMINA_CPU} cpu. "
                    f"PDB -> protein-row mapping: {json.dumps(pdb_map)}.\n")
            if MAX_LIGANDS_PER_CELL:
                f.write(f"\n**Partial**: capped at {MAX_LIGANDS_PER_CELL} ligands per "
                        f"protein-run.\n")
            if len(runs) < 10:
                f.write(f"\n**Partial**: {len(runs)} of 10 runs docked.\n")
        print(f"\nappended to {os.path.join(EVAL_DIR, 'report.md')}")
else:
    print("\nNothing docked.")


Nothing docked.
